In [1]:
# Cell 1: Install required libraries

!pip -q install -U transformers datasets evaluate accelerate rouge_score sentencepiece

In [2]:
# Cell 2: Check the runtime environment

import os
import platform
import torch

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
    print("bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU detected. Please switch the runtime to GPU in Colab.")

Python version: 3.12.13
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU name: NVIDIA A100-SXM4-40GB
CUDA device count: 1
bf16 supported: True


In [3]:
# Cell 3: Import libraries and set random seeds

import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import evaluate
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [4]:
# Cell 4: Define file paths and experiment configuration

TRAIN_PATH = "/train.json"
VAL_PATH = "/val.json"
TEST_PATH = "/test.json"

MODEL_CHECKPOINT = "facebook/mbart-large-50-many-to-many-mmt"
OUTPUT_DIR = "./mbart_large_50_dialogue_summarization"

SRC_LANG = "en_XX"
TGT_LANG = "en_XX"

MAX_SOURCE_LENGTH = 768
MAX_TARGET_LENGTH = 64

NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 2e-5
NUM_BEAMS = 5

PER_DEVICE_TRAIN_BATCH_SIZE = 4
PER_DEVICE_EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 6

print("Model checkpoint:", MODEL_CHECKPOINT)
print("Output directory:", OUTPUT_DIR)
print("Source language:", SRC_LANG)
print("Target language:", TGT_LANG)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

Model checkpoint: facebook/mbart-large-50-many-to-many-mmt
Output directory: ./mbart_large_50_dialogue_summarization
Source language: en_XX
Target language: en_XX
Effective train batch size: 24


In [5]:
# Cell 5: Define helper functions for loading JSON files

def load_json_file(file_path: str) -> pd.DataFrame:
    """Load a JSON file that may be either a JSON array or JSON Lines."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            return pd.DataFrame(data)
        if isinstance(data, dict):
            return pd.DataFrame([data])
        raise ValueError(f"Unsupported JSON structure in {file_path}")
    except json.JSONDecodeError:
        records = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.DataFrame(records)

train_df = load_json_file(TRAIN_PATH)
val_df = load_json_file(VAL_PATH)
test_df = load_json_file(TEST_PATH)

print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("test_df shape:", test_df.shape)

train_df shape: (14732, 4)
val_df shape: (818, 4)
test_df shape: (819, 4)


In [6]:
# Cell 6: Inspect the raw data

display(train_df.head(2))
print("Columns:", list(train_df.columns))

,dialogue,summary,summary_de,summary_zh
0,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...,amanda hat kekse gebacken und wird jerry morge...,阿曼达烤了饼干，明天会给杰瑞带一些。
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...,olivia und olivier stimmen bei dieser wahl für...,奥利维亚和奥利维尔在这次选举中要投票给自由党。


Columns: ['dialogue', 'summary', 'summary_de', 'summary_zh']


In [7]:
# Cell 7: Keep only the required columns

required_columns = ["dialogue", "summary"]

train_df = train_df[required_columns].copy()
val_df = val_df[required_columns].copy()
test_df = test_df[required_columns].copy()

print("Remaining columns:", list(train_df.columns))

Remaining columns: ['dialogue', 'summary']


In [8]:
# Cell 8: Clean and preprocess the text

def clean_dialogue_text(text: str) -> str:
    """Clean the dialogue text while preserving useful speaker information."""
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove placeholder tokens such as <file_gif>, <file_photo>, and similar markers.
    text = re.sub(r"<file_[^>]+>", " ", text)

    # Normalize whitespace.
    text = text.replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)

    return text.strip()

def clean_summary_text(text: str) -> str:
    """Clean the target summary text."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

for df in [train_df, val_df, test_df]:
    df["dialogue"] = df["dialogue"].apply(clean_dialogue_text)
    df["summary"] = df["summary"].apply(clean_summary_text)

    # Remove empty rows.
    df.dropna(subset=["dialogue", "summary"], inplace=True)
    df = df[(df["dialogue"].str.len() > 0) & (df["summary"].str.len() > 0)]

# Re-apply the empty-row filtering correctly after inplace operations.
train_df = train_df[(train_df["dialogue"].str.len() > 0) & (train_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)
val_df = val_df[(val_df["dialogue"].str.len() > 0) & (val_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)
test_df = test_df[(test_df["dialogue"].str.len() > 0) & (test_df["summary"].str.len() > 0)].drop_duplicates().reset_index(drop=True)

print("train_df shape after cleaning:", train_df.shape)
print("val_df shape after cleaning:", val_df.shape)
print("test_df shape after cleaning:", test_df.shape)

display(train_df.head(2))

train_df shape after cleaning: (14730, 2)
val_df shape after cleaning: (818, 2)
test_df shape after cleaning: (819, 2)


,dialogue,summary
0,Amanda: I baked cookies. Do you want some? \nJ...,Amanda baked cookies and will bring Jerry some...
1,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...


In [9]:
# Cell 9: Load the tokenizer and inspect token length distributions

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
)

def get_token_lengths(texts, tokenizer):
    return [len(tokenizer(text, truncation=False)["input_ids"]) for text in texts]

sample_size_for_stats = min(2000, len(train_df))
sample_dialogues = train_df["dialogue"].iloc[:sample_size_for_stats].tolist()
sample_summaries = train_df["summary"].iloc[:sample_size_for_stats].tolist()

dialogue_lengths = get_token_lengths(sample_dialogues, tokenizer)
summary_lengths = get_token_lengths(sample_summaries, tokenizer)

length_stats = pd.DataFrame(
    {
        "dialogue_tokens": pd.Series(dialogue_lengths),
        "summary_tokens": pd.Series(summary_lengths),
    }
)

display(length_stats.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

print("Current MAX_SOURCE_LENGTH:", MAX_SOURCE_LENGTH)
print("Current MAX_TARGET_LENGTH:", MAX_TARGET_LENGTH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


,dialogue_tokens,summary_tokens
count,2000.000000,2000.000000
mean,143.235000,29.369000
std,104.990029,15.048491
min,14.000000,3.000000
50%,114.500000,26.000000
90%,285.100000,51.000000
95%,361.050000,60.000000
99%,500.030000,72.010000
max,687.000000,83.000000


Current MAX_SOURCE_LENGTH: 768
Current MAX_TARGET_LENGTH: 64


In [10]:
# Cell 10: Convert pandas DataFrames to Hugging Face Dataset objects

dataset_dict = DatasetDict(
    {
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    }
)

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'summary'],
        num_rows: 14730
    })
    validation: Dataset({
        features: ['dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['dialogue', 'summary'],
        num_rows: 819
    })
})

In [11]:
# Cell 11: Define the tokenization function

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="Tokenizing datasets",
)

tokenized_datasets

Tokenizing datasets:   0%|          | 0/14730 [00:00<?, ? examples/s]

Tokenizing datasets:   0%|          | 0/818 [00:00<?, ? examples/s]

Tokenizing datasets:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14730
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [12]:
# Cell 12: Load the model, data collator, and ROUGE metric

import numpy as np
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # Some versions return a tuple, while others return a single ndarray.
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.array(predictions)
    labels = np.array(labels)

    # If predictions are logits instead of generated token ids,
    # convert them to token ids with argmax.
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Make sure the dtype is valid for decoding.
    predictions = predictions.astype(np.int64)

    # Replace invalid negative ids before decoding.
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions)

    # Replace label masking id (-100) with the pad token id.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    # ROUGE-LSum prefers newline-separated sentence boundaries.
    decoded_preds = ["\n".join(pred.split(". ")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.split(". ")) for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True,
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}

    prediction_lens = [
        np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions
    ]
    result["gen_len"] = round(float(np.mean(prediction_lens)), 4)

    return result

In [13]:
# New Cell: Reload a fresh model before training

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Fresh model loaded from:", MODEL_CHECKPOINT)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Fresh model loaded from: facebook/mbart-large-50-many-to-many-mmt


In [14]:
# Cell 13: Prepare mixed precision settings for Colab A100

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

print("use_bf16:", use_bf16)
print("use_fp16:", use_fp16)

use_bf16: True
use_fp16: False


In [15]:
# New Cell: Reload a fresh mBART model before training

from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, GenerationConfig

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

model.generation_config = GenerationConfig.from_model_config(model.config)
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

print("Fresh model loaded from:", MODEL_CHECKPOINT)
print("forced_bos_token_id:", model.generation_config.forced_bos_token_id)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

Fresh model loaded from: facebook/mbart-large-50-many-to-many-mmt
forced_bos_token_id: 250004


In [17]:
# Cell 14: Define training arguments and trainer

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=NUM_BEAMS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeLsum",
    greater_is_better=True,
    optim="adafactor",
    bf16=use_bf16,
    fp16=use_fp16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [18]:
# Cell 15: Start full training

train_result = trainer.train()
train_result

`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,9.074069,1.470396,49.810900,25.620200,41.245100,45.814700,26.481700
2,7.816922,1.444322,50.960300,26.734600,42.137000,46.627800,28.767700
3,6.847761,1.457901,50.729100,25.503300,41.231100,45.893200,28.372900
4,6.184221,1.491100,50.646600,25.778400,40.983400,46.156900,29.152800
5,5.515695,1.518657,50.828300,25.601900,41.026400,46.266200,29.013400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=3070, training_loss=7.186211897262921, metrics={'train_runtime': 5034.9756, 'train_samples_per_second': 14.628, 'train_steps_per_second': 0.61, 'total_flos': 4.005735361078886e+16, 'train_loss': 7.186211897262921, 'epoch': 5.0})

In [19]:
# Cell 16: Evaluate on the validation set with predict()

try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.pop_callback(NotebookProgressCallback)
except Exception:
    pass

validation_output = trainer.predict(tokenized_datasets["validation"])
validation_metrics = validation_output.metrics
validation_metrics

{'test_loss': 1.4443222284317017,
 'test_rouge1': 50.9603,
 'test_rouge2': 26.7346,
 'test_rougeL': 42.137,
 'test_rougeLsum': 46.6278,
 'test_gen_len': 28.7677,
 'test_runtime': 221.825,
 'test_samples_per_second': 3.688,
 'test_steps_per_second': 0.924}

In [20]:
# Cell 17: Evaluate on the test set with predict()

test_output = trainer.predict(tokenized_datasets["test"])
test_metrics = test_output.metrics
test_metrics

{'test_loss': 1.4613215923309326,
 'test_rouge1': 48.9213,
 'test_rouge2': 24.1386,
 'test_rougeL': 40.3879,
 'test_rougeLsum': 44.7998,
 'test_gen_len': 28.5165,
 'test_runtime': 216.1515,
 'test_samples_per_second': 3.789,
 'test_steps_per_second': 0.948}

In [21]:
# Cell 18: Save the final model and tokenizer

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved model and tokenizer to:", OUTPUT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model and tokenizer to: ./mbart_large_50_dialogue_summarization


In [22]:
# Cell 19: Test the fine-tuned model on the provided sample dialogue

sample_dialogue = """Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye"""

sample_dialogue_clean = clean_dialogue_text(sample_dialogue)

inputs = tokenizer(
    sample_dialogue_clean,
    return_tensors="pt",
    max_length=MAX_SOURCE_LENGTH,
    truncation=True,
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

generated_ids = model.generate(
    **inputs,
    max_length=64,
    min_length=12,
    num_beams=5,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    early_stopping=True,
    forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG],
)

generated_summary = tokenizer.decode(
    generated_ids[0],
    skip_special_tokens=True,
)

print("Input dialogue:")
print(sample_dialogue_clean)
print("\nGenerated summary:")
print(generated_summary)

Input dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: 
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: 
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Generated summary:
Betty called Larry last time they were at the park together. Amanda has Betty's number.


In [ ]:
# Optional Cell: Load the uploaded model from the Hugging Face Hub

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

REPO_ID = "yunu919/bart-large-dialogue-summarization"

tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(REPO_ID)